# 00 · 第一个 Notebook

**怎么看这个文件**：这不是一个网页，是一个"能运行的笔记本"。

- 每一块灰色/白色的框叫 **cell（单元）**，里面是代码
- 点中某个 cell，按 **Shift + Enter** 运行它，结果就显示在它下面
- 从上往下依次运行，变量会一直留在内存里

> 记住一句话：**Notebook = 分块运行的 Python，跑过的东西不会丢。**

In [1]:
# 第 1 格：确认显卡能用
import torch

print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
print("显卡:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"显存: {total/1024**3:.2f} GB（空闲 {free/1024**3:.2f} GB）")

PyTorch: 2.9.1+cu126
CUDA 可用: True
显卡: NVIDIA GeForce RTX 3060 Laptop GPU
显存: 6.00 GB（空闲 5.01 GB）


## 为什么要用 Notebook 而不是普通脚本

加载一个模型要几秒到几分钟。如果用 `.py` 脚本，每改一次 prompt 都要重新加载一遍模型。

在 Notebook 里：**加载一次，之后随便改、随便试**——模型一直在内存里。

这就是它在大模型工作里不可替代的原因。

In [2]:
# 第 2 格：加载模型（只跑一次，之后一直用）
# 首次运行会下载约 1.2 GB，走的是国内镜像
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
model.eval()
print("加载完成，参数量", round(sum(p.numel() for p in model.parameters()) / 1e6), "M")

D:\dev\conda-envs\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1026 19:31:31.138000 22460 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   1%|          | 2/311 [00:00<00:15, 19.56it/s]

Loading weights:  10%|█         | 32/311 [00:00<00:01, 174.16it/s]

...

Loading weights:  83%|████████▎ | 259/311 [00:01<00:00, 150.04it/s]

Loading weights:  89%|████████▊ | 276/311 [00:01<00:00, 152.05it/s]

Loading weights:  95%|█████████▍| 295/311 [00:01<00:00, 161.00it/s]

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 158.76it/s]

加载完成，参数量 596 M


In [3]:
# 第 3 格：生成（改 prompt 直接重跑这一格，不用重新加载模型）
def ask(prompt, max_new_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print(ask("用一句话解释什么是注意力机制："))

注意力机制是一种使模型能够关注输入中重要部分并高效处理信息的技术。


In [4]:
# 第 4 格：看显存占用（每次都养成看的习惯）
print(f"当前分配: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"峰值占用: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

当前分配: 1.12 GB
峰值占用: 1.55 GB


---
## 你已经会了三件事

1. 分格运行代码（Shift + Enter）
2. 模型只加载一次，反复使用
3. 随时查看显存

**下一步**：把 `ask()` 换几个 prompt 试试，观察回答和速度变化。

> 常见坑：如果报 `NameError: model is not defined`，说明第 2 格没跑——**Notebook 不会自动帮你跑前面的格子**。